In [ ]:
import sys; sys.path.append('..')
import MeshFEM, mesh, mesh_energy, param_utils, viewer, benchmark
import numpy as np
import sim_utils

import matplotlib
from matplotlib import pyplot as plt
import visualization

import newton_flow, newton_flow_utils as nfu

In [ ]:
m = param_utils.load('../../models/lucy.msh.xz')
tutte_uv = param_utils.tutteInitialization(m)
v = mesh_energy.NodalVars(m, 2)
m_2d = mesh.Mesh(tutte_uv, m.elements())
nf = newton_flow.symmetric_dirichlet(m_2d, v)

In [ ]:
v.setVars(tutte_uv.ravel())

In [ ]:
nf.setRestVertexPositions(m.vertices())

In [ ]:
init_scale = 299.12122825287764
v.setVars(init_scale * tutte_uv.ravel())

In [ ]:
nf.projectionSmoothingEpsilon = 0 # 1e-4 # 1e-8

In [ ]:
import py_newton_optimizer
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(v, [nf])

In [ ]:
# Nullspace pinning strategy
FIX_VARS = False
if FIX_VARS:
    import elastic_solid, energy
    es = elastic_solid.ElasticSolid(m_2d, energy.CommonNeoHookeanYoungPoisson(2, 1, 0.3))
    pin_vars, _ = es.prepareRigidMotionPins()
    v.setVars(init_scale * es.getVars())
    prob.setFixedVars(pin_vars)
else:
    # prob.hessianShift = 1e-5
    # nf.elementHessianShift = 1e-8
    # nf.elementHessianShift = 1e-8
    prob.hessianShift = 1e-10
    prob.useRelativeHessianShift = False

In [ ]:
import newton_flow_utils

In [ ]:
constant_speed = True
always_project = False

In [ ]:
opt = prob.optimizer()
opt.options.factorizer = opt.options.factorizer.CatamariAMD
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
if always_project: opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()

In [ ]:
import flip_avoiding_step_length
prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m_2d.elements())
prob.initialFeasibleStepLengthComputer.backoffFactor = 0.95

In [ ]:
fasl = flip_avoiding_step_length.FlipAvoidingStepLength(m_2d.elements())

In [ ]:
np.linalg.norm(prob.gradient())

In [ ]:
fv = nfu.ground_truth_flow(opt, 1.0, verbose=True, grad_tol=1e-4, step_limiter = fasl, max_iters=100)

In [ ]:
np.sort(nf.elementHessianMinimumEigenvalues())

In [ ]:
opt.options.niter = 0
opt.optimize()
opt.update_factorizations()

In [ ]:
always_project = True
opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAdaptive()
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
if always_project: opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()

In [ ]:
import importlib
importlib.reload(visualization);
importlib.reload(nfu);

extrapolation_dist = 3
constant_speed = True
# constant_speed = False
num_frames = min(500, len(fv))

methods = [
    (1, nfu.eval_trajectory_taylor, 'Newton'),
    # (2, nfu.eval_trajectory_logspiral, 'Deg 2 spiral'),
    (2, nfu.eval_trajectory_taylor, 'Deg 2 Taylor'),
    (3, nfu.eval_trajectory_taylor, 'Deg 3 Taylor'),
    # (4, nfu.eval_trajectory_taylor, 'Deg 4 Taylor'),
    (5, nfu.eval_trajectory_taylor, 'Deg 5 Taylor'),
    (14, nfu.eval_trajectory_vector_pade, 'Pade 14'),
    (19, nfu.eval_trajectory_vector_pade, 'Pade 19'),
]
ff = lambda i:  visualization.flow_frame(i, opt, fv, extrapolation_dist, constant_speed, corners_only=True,
                         extrapolation_method_list=methods, truncate=True)

In [ ]:
frame = 24

In [ ]:
# Experiment with altering the eigenvalue clamp target to progressively disable projection.
# This seems to help closer to the optimum. Globally scaling the projection
# (via `eigenvalueProjectionModulation`) does not appear to be beneficial.
benchmark.reset()
prob.setVars(fv[frame].ravel())
nf.eigenvalueClampTarget = 0
# nf.eigenvalueClampTarget = -4
nf.eigenvalueProjectionModulation = 1.0
d = opt.newton_step()
sl = fasl.eval(prob.getVars(), d)
print(sl * np.linalg.norm(d), np.linalg.norm(d), sl)
print(f"Factorizations:\t{benchmark.numInvocations('Factorize$')}")

In [ ]:
benchmark.reset()
extrapolation_dist = 6
ff(frame)
plt.show()
benchmark.report()